In [1]:
%env CUDA_VISIBLE_DEVICES = 3

env: CUDA_VISIBLE_DEVICES=3


In [2]:
from fetch_dataset_and_processing import find_train_val_files, find_test_files, ImageNet
from train_loop import train
from inference import predict, predict_one_sample, make_predictions
from visualizing import imshow, plot_train_set, plot_results

from resnet50 import IdentityBlock, ConvBlock, ResNet50
from resnet18_cnn import ResBlock, ResNet18

import torch
from sklearn.model_selection import train_test_split
import os
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
from torch.utils.data import Dataset, DataLoader

In [3]:
! nvidia-smi

Thu Feb  2 14:18:23 2023       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 520.56.06    Driver Version: 520.56.06    CUDA Version: 11.8     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  NVIDIA GeForce ...  Off  | 00000000:1A:00.0 Off |                  N/A |
| 28%   31C    P0    46W / 260W |      0MiB / 11264MiB |      0%      Default |
|                               |                      |                  N/A |
+-------------------------------+----------------------+----------------------+
|   1  NVIDIA GeForce ...  Off  | 00000000:1B:00.0 Off |                  N/A |
| 27%   

In [3]:
device = 'cuda:0'
device

'cuda:0'

In [4]:
train_val_files, train_val_labels = find_train_val_files()
test_files, test_labels = find_test_files()

In [5]:
len(train_val_files), len(test_files)

(9103, 3791)

In [6]:
train_set, val_set, tr_labels, val_labels = train_test_split(train_val_files, train_val_labels, test_size=0.2, \
                                          stratify=train_val_labels)

In [7]:
train_dataset = ImageNet(train_set, tr_labels, 'train')
val_dataset = ImageNet(val_set, val_labels, 'val')
test_dataset = ImageNet(test_files, test_labels, 'test')

In [8]:
batch_size = 16
train_loader = DataLoader(train_dataset, batch_size=batch_size)
val_loader = DataLoader(val_dataset, batch_size=batch_size)

### ResNet 50 (with trick)

In [5]:
resnet = ResNet50(ConvBlock, IdentityBlock, 'A').to(device)
resnet

ResNet50(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(4, 4))
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU()
  (pool): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  (linear): Linear(in_features=2048, out_features=10, bias=True)
  (conv_block1): ConvBlock(
    (conv_block): Sequential(
      (0): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1))
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU()
      (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (5): ReLU()
      (6): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1))
      (7): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (8): ReLU()
    )
    (shortcut): Sequential(
      (0): AvgPool2d(kernel_size=3, st

### Training

In [36]:
epochs = 20
opt = torch.optim.Adam(resnet.parameters(), lr=1e-4)
loss_fn = nn.CrossEntropyLoss()
path = 'ResNet_exp/Resnet_50_20ep.pth'
name = 'ResNet_exp/resnet_50_20ep'

In [37]:
losses_res, metrics_res = train(resnet, opt, loss_fn, epochs, train_loader, val_loader, name, path)

Epoch 0/19:


train iter:: 100%|██████████| 456/456 [02:35<00:00,  2.94it/s]


train Loss: 1.7844
train Accuracy: 0.3921


val iter:: 100%|██████████| 114/114 [00:15<00:00,  7.20it/s]


val Loss: 1.5039
val Accuracy: 0.5212
Epoch 1/19:


...

val Loss: 0.7807
val Accuracy: 0.7860
Epoch 19/19:


train iter:: 100%|██████████| 456/456 [02:30<00:00,  3.03it/s]


train Loss: 0.2881
train Accuracy: 0.9206


val iter:: 100%|██████████| 114/114 [00:15<00:00,  7.13it/s]


val Loss: 0.7807
val Accuracy: 0.7860


### Inference

In [9]:
resnet = ResNet50(ConvBlock, IdentityBlock, 'A').to(device)
path = 'ResNet_exp/Resnet_50_20ep.pth'
resnet.load_state_dict(torch.load(path))

<All keys matched successfully>

In [10]:
idxs = range(len(test_dataset)) 
imgs = [test_dataset[id][0].unsqueeze(0) for id in idxs]
actual_labels = [test_dataset[id][1] for id in idxs]

In [12]:
predictions = make_predictions(resnet, imgs)
acc = accuracy_score(predictions, actual_labels)

print(f'The accuracy on the test set of ResNet50 is {round(acc, 4)}')

The accuracy on the test set of ResNet50 is 0.7713
